# Sentiment Analysis on Tweets

In this lab, we classify tweets into three categories: **negative**, **neutral** or **positive**.

We use the [TweetEval](https://huggingface.co/datasets/cardiffnlp/tweet_eval) dataset (*sentiment* task): about 60,000 English tweets, annotated by hand.

Outline:

1. prepare batches without padding with PyTorch's `PackedSequence`;
2. train a bidirectional LSTM with PyTorch Lightning, with early stopping and saving of the best model;
3. compare the result with a TF-IDF baseline and with pretrained transformers: an encoder we fine-tune ourselves, and a model already fine-tuned on this task.

*Remember to enable the GPU: Runtime > Change runtime type > GPU.*

## Imports

In [ ]:
!pip install -q datasets lightning torchmetrics

In [ ]:
import collections

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import spacy
import torch
from datasets import load_dataset
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, recall_score
from torch import nn
from torch.nn.utils.rnn import PackedSequence, pack_sequence
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import MulticlassRecall

lightning.seed_everything(0)

## Loading the data

Hugging Face's `datasets` library downloads the dataset already split into training, validation and test sets.

In [ ]:
tweets = load_dataset("cardiffnlp/tweet_eval", "sentiment")
label_names = ["negative", "neutral", "positive"]
print(tweets)

The next cell shows ten training tweets with their label, the class distribution and the length of the tweets (in words).

In [ ]:
train_df = tweets["train"].to_pandas()
train_df["sentiment"] = train_df.label.map(dict(enumerate(label_names)))
print(train_df.sample(10, random_state=0)[["sentiment", "text"]].to_string())

train_df.sentiment.value_counts().plot.bar(title="Class distribution")
plt.show()

train_df.text.str.split().str.len().plot.hist(bins=30, title="Tweet length (words)")
plt.show()

*How were user names handled? Are the classes balanced, and what does it imply for the choice of metric? What is the typical length of a tweet?*

### Solution

- User names are replaced with `@user`: the tweets are anonymized.
- The *neutral* class is the most frequent, the *negative* class the rarest: accuracy would favor a model that mostly predicts *neutral*. TweetEval's official metric is the **per-class average recall** (*macro recall*), which gives the same weight to each class.
- A tweet is about twenty words long, rarely more than 40.

## Baseline: TF-IDF and logistic regression

Before training a neural network, let's measure what a simple model achieves: a logistic regression on a TF-IDF representation of the tweets. If the network doesn't do better, it's useless!

In [ ]:
vectorizer = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(tweets["train"]["text"])
X_val_tfidf = vectorizer.transform(tweets["validation"]["text"])

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_tfidf, tweets["train"]["label"])
baseline_recall = recall_score(tweets["validation"]["label"],
                               baseline.predict(X_val_tfidf), average="macro")
print(f"TF-IDF macro recall: {baseline_recall:.3f}")

## Tokenization with spaCy

For a recurrent network, each tweet must become a sequence of word indices. We use spaCy's tokenizer, which handles punctuation, emoticons and contractions much better than a simple `split()`.

`spacy.blank("en")` creates an empty English pipeline: it only contains the tokenizer, which is very fast.

In [ ]:
nlp = spacy.blank("en")


def tokenize(text: str) -> list[str]:
  return [token.lower_ for token in nlp.tokenizer(text) if not token.is_space]


print(tokenize(tweets["train"][0]["text"]))

## Building the vocabulary

The vocabulary is built from the training set **only**, with the words seen at least twice; index `0` is reserved for unknown words (`<unk>`). `numericalize` turns a tweet into a list of indices.

In [ ]:
counts = collections.Counter(
    word for text in tweets["train"]["text"] for word in tokenize(text))
vocab = {"<unk>": 0}
for word, count in counts.most_common():
  if count >= 2:
    vocab[word] = len(vocab)
print(f"Vocabulary size : {len(vocab)}")


def numericalize(text: str) -> list[int]:
  return [vocab.get(word, vocab["<unk>"]) for word in tokenize(text)]


print(numericalize(tweets["train"][0]["text"]))

*Why not use the validation and test sets to build the vocabulary?*

### Solution

The vocabulary is part of the model: building it with validation or test data would "leak" information about these data, and the evaluation would be too optimistic.

## Batches without padding: `PackedSequence`

Tweets don't all have the same length. Rather than padding the sequences of a batch, PyTorch can **pack** them into a `PackedSequence`: all the tokens of the batch are laid end to end, time step by time step, without any padding.

- `data`: first the first token of each sequence, then the second token of each sequence that has at least two, etc.;
- `batch_sizes`: the number of sequences still running at each time step;
- `sorted_indices` / `unsorted_indices`: the order (by decreasing length) in which the sequences are laid out, and how to get back to the original order.

PyTorch's recurrent layers (`nn.RNN`, `nn.GRU`, `nn.LSTM`) accept a `PackedSequence` directly: at each time step, they only compute on the sequences still running, and their last hidden state `h_n` is the one of each sequence's last *real* token, put back in the original order.

*Run the next cell and work out how `data` and `batch_sizes` are built from the three sequences.*

In [ ]:
example = pack_sequence([torch.tensor([1, 2, 3]), torch.tensor([4, 5]),
                         torch.tensor([6, 7, 8, 9])], enforce_sorted=False)
print(example)

## `Dataset` and `DataLoader`

`TweetDataset` returns each tweet as a tensor of indices, with its label.

In [ ]:
class TweetDataset(Dataset):
  def __init__(self, split) -> None:
    # A tweet without any token becomes a single <unk>: an empty sequence
    # can't be packed
    self.sequences = [torch.tensor(numericalize(text) or [vocab["<unk>"]])
                      for text in split["text"]]
    self.labels = split["label"]

  def __len__(self) -> int:
    return len(self.labels)

  def __getitem__(self, i: int) -> tuple[torch.Tensor, int]:
    return self.sequences[i], self.labels[i]


train_dataset = TweetDataset(tweets["train"])
val_dataset = TweetDataset(tweets["validation"])
test_dataset = TweetDataset(tweets["test"])
print(train_dataset[0])

*Write the `collate(batch)` function the `DataLoader`s will use to form their batches: it receives a list of `(tensor of indices, label)` pairs and returns `(packed sequences, labels)`. Use `pack_sequence(..., enforce_sorted=False)`: the sequences don't have to be sorted by length, `pack_sequence` takes care of it.*

In [ ]:
# Your code here

### Solution

In [ ]:
def collate(batch: list[tuple[torch.Tensor, int]]) -> tuple[PackedSequence, torch.Tensor]:
  sequences, labels = zip(*batch)
  return pack_sequence(sequences, enforce_sorted=False), torch.tensor(labels)

### Packed batches

The `DataLoader`s form their batches with `collate`. On a training batch, let's compare the number of packed tokens with the size the same batch would have with padding:

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=64, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=64, collate_fn=collate)

packed, labels = next(iter(train_loader))
# Number of tweets × length of the longest
padded_size = len(labels) * len(packed.batch_sizes)
print(f"Batch tokens: {packed.data.numel()} packed against {padded_size} "
      f"with padding, i.e. {1 - packed.data.numel() / padded_size:.0%} of padding avoided")

The lengths of the tweets in a batch vary a lot: with padding, a batch would hold about twice as many positions as real tokens, and the LSTM would do as many useless computations. The `PackedSequence` avoids them, and the model no longer needs a padding index.

## The model: a bidirectional LSTM

*Complete the following model, which takes a `PackedSequence` of word indices as input: embeddings, a bidirectional LSTM, and an output layer that computes one logit per class from the last hidden states of both directions. Add a dropout of 0.3 on the embeddings and before the output layer.*

*`nn.Embedding` and `nn.Dropout` don't accept a `PackedSequence`: apply them to the tokens `packed.data`, then rebuild a `PackedSequence` with `packed._replace(data=...)`, which keeps `batch_sizes` and the sorting indices.*

In [ ]:
class SentimentLSTM(nn.Module):
  def __init__(self, vocab_size: int, embedding_dim: int = 128,
               hidden_size: int = 128, n_classes: int = 3) -> None:
    super().__init__()
    # Your code here

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    # Your code here
    pass

### Solution

In [ ]:
class SentimentLSTM(nn.Module):
  def __init__(self, vocab_size: int, embedding_dim: int = 128,
               hidden_size: int = 128, n_classes: int = 3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.lstm = nn.LSTM(embedding_dim, hidden_size, bidirectional=True)
    self.dropout = nn.Dropout(0.3)
    self.head = nn.Linear(2 * hidden_size, n_classes)

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    # Embeddings and dropout of all the batch's tokens at once, put back in a
    # PackedSequence with the same structure
    embedded = packed._replace(data=self.dropout(self.embedding(packed.data)))
    _, (h_n, _) = self.lstm(embedded)
    # Last hidden states of the forward and backward directions, in the tweets'
    # original order
    summary = torch.cat([h_n[-2], h_n[-1]], dim=1)
    return self.head(self.dropout(summary))


lstm = SentimentLSTM(len(vocab))
print(lstm)
print(sum(p.numel() for p in lstm.parameters()), "parameters")

Why `h_n[-2]` and `h_n[-1]`: what `h_n` holds at the end of a multi-layer, bidirectional GRU or LSTM, as a video: [`h_n` layer by layer](https://shuuchuu.github.io/videos/nn/en/gru-final-hidden-state.mp4).

## Training

The `LightningModule` below describes the loss function, the metric (the macro recall) and the optimizer. Batches are `(PackedSequence, labels)` pairs: the `Trainer` moves them to the GPU itself, `PackedSequence` included.

In [ ]:
class SentimentClassifier(lightning.LightningModule):
  """Lightning wrapper: classify the sentiment of packed tweets."""

  def __init__(self, model: nn.Module, learning_rate: float = 1e-3,
               n_classes: int = 3) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # torchmetrics needs one metric instance per stage
    self.recalls = nn.ModuleDict({
        f"{stage}_recall": MulticlassRecall(num_classes=n_classes, average="macro")
        for stage in ("val", "test")})

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    return self.model(packed)

  def _step(self, batch: tuple[PackedSequence, torch.Tensor],
            stage: str) -> torch.Tensor:
    packed, labels = batch
    logits = self(packed)
    loss = nn.functional.cross_entropy(logits, labels)
    # Without batch_size, Lightning would infer it from packed.data: the number of tokens
    self.log(f"{stage}_loss", loss, batch_size=len(labels), on_step=False,
             on_epoch=True, prog_bar=True)
    if stage != "train":
      recall = self.recalls[f"{stage}_recall"]
      recall(logits, labels)
      self.log(f"{stage}_recall", recall, batch_size=len(labels), on_step=False,
               on_epoch=True, prog_bar=True)
    return loss

  def training_step(self, batch: tuple[PackedSequence, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[PackedSequence, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: tuple[PackedSequence, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(self, batch: tuple[PackedSequence, torch.Tensor],
                   batch_index: int) -> torch.Tensor:
    packed, _ = batch
    return self(packed).argmax(dim=1)

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)


def plot_metrics(trainer: lightning.Trainer) -> None:
  """Plot the losses and the validation macro recall over the epochs."""
  # Training and validation metrics aren't written on the same lines of the
  # CSV file: we group them by epoch
  metrics = pandas.read_csv(f"{trainer.logger.log_dir}/metrics.csv").groupby(
      "epoch").mean(numeric_only=True).dropna(subset=["train_loss", "val_loss"])
  metrics[["train_loss", "val_loss"]].plot(title="Loss", marker="o")
  metrics[["val_recall"]].plot(title="Validation macro recall", marker="o", legend=False)
  plt.show()

*Train the model for at most 10 epochs, with two callbacks:*

- *`EarlyStopping`, which stops training when `val_recall` doesn't improve for 2 epochs;*
- *`ModelCheckpoint`, which keeps the best model's weights according to `val_recall`.*

*Then plot the learning curves with `plot_metrics`, which reads the metrics of a `CSVLogger`.*

In [ ]:
# Your code here

### Solution

In [ ]:
model = SentimentClassifier(SentimentLSTM(len(vocab)))
checkpoint = ModelCheckpoint(monitor="val_recall", mode="max")
trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name="sentiment_lstm"),
    callbacks=[EarlyStopping(monitor="val_recall", mode="max", patience=2),
               checkpoint])
trainer.fit(model, train_loader, val_loader)
print(f"Best validation macro recall: {checkpoint.best_model_score:.3f}")
plot_metrics(trainer)

## Final evaluation on the test set

*Evaluate the best model on the test set: its macro recall, then the classification report and the confusion matrix of its predictions, with the `show_evaluation` function below. Do the same for the TF-IDF baseline.*

*Which classes are most often confused? Does the LSTM beat the baseline?*

In [ ]:
def show_evaluation(truths: numpy.ndarray, predictions: numpy.ndarray,
                    title: str) -> None:
  """Show the classification report and the confusion matrix."""
  print(title)
  print(classification_report(truths, predictions, target_names=label_names,
                              digits=3))
  ConfusionMatrixDisplay.from_predictions(truths, predictions,
                                          display_labels=label_names,
                                          normalize="true")
  plt.title(title)
  plt.show()

In [ ]:
# Your code here

### Solution

In [ ]:
trainer.test(model, test_loader, ckpt_path="best")

# trainer.predict returns the list of predict_step's outputs, one per batch
predictions = torch.cat(trainer.predict(model, test_loader, ckpt_path="best")).numpy()
truths = numpy.array(tweets["test"]["label"])
show_evaluation(truths, predictions, "Bidirectional LSTM")

baseline_predictions = baseline.predict(vectorizer.transform(tweets["test"]["text"]))
show_evaluation(truths, baseline_predictions, "TF-IDF baseline")

- Confusions mostly involve the *neutral* class; *negative* and *positive* are rarely confused with each other.
- The LSTM trained from scratch is only slightly better than the TF-IDF baseline (macro recall of about 0.59 against 0.56 on the test set): with only a few tens of thousands of tweets, it lacks data to learn the meaning of words. This is the motivation for pretrained models.

## For comparison: a pretrained transformer

The [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest) model is a RoBERTa pretrained on millions of tweets, then fine-tuned on this same task. The `transformers` `pipeline` function makes it usable in one line.

*Evaluate it on the first 1,000 test tweets and compare its macro recall with the LSTM's on the same tweets. What do you conclude?*

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",
                      model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                      device=0 if torch.cuda.is_available() else -1)
print(classifier("I love this course on PyTorch!"))

In [ ]:
# Your code here

### Solution

In [ ]:
subset = tweets["test"].select(range(1000))
name_to_label = {"negative": 0, "neutral": 1, "positive": 2}
roberta_predictions = [name_to_label[output["label"]]
                       for output in classifier(list(subset["text"]), batch_size=64)]
print(f"Pretrained RoBERTa: "
      f"{recall_score(list(subset['label']), roberta_predictions, average='macro'):.3f}")
print(f"LSTM: {recall_score(truths[:1000], predictions[:1000], average='macro'):.3f}")

The pretrained transformer does much better (about 0.71 against 0.59): it learned the language of tweets from millions of examples before being fine-tuned. For a common text classification task, starting from a pretrained model is today's first reflex; training an LSTM from scratch remains useful under tight compute constraints, or when no pretrained model exists for the language or domain.

## Fine-Tuning a Pretrained Encoder Ourselves

cardiffnlp's RoBERTa was fine-tuned on this task by others. Let's do it ourselves, from a general-purpose encoder that has never seen these labels: [`distilroberta-base`](https://huggingface.co/distilbert/distilroberta-base), a smaller version of RoBERTa (82 M parameters). `AutoModelForSequenceClassification` adds a 3-output classification head to it (see the "Fine-Tuning a Pretrained Encoder" slide), and the whole model is trained with a small learning rate.

The next cell tokenizes the tweets with the encoder's tokenizer and defines the Lightning wrapper.

*Train this model for 2 epochs, keeping the best one according to `val_recall`, evaluate it on the test set with `show_evaluation`, then compare its macro recall on the first 1,000 test tweets with those of the LSTM and of cardiffnlp's RoBERTa.*

In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding)

ENCODER = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
# Tokenize each tweet: subword identifiers and attention mask
encoded = tweets.map(
    lambda batch: tokenizer(batch["text"], truncation=True, max_length=128),
    batched=True).remove_columns(["text"]).rename_column("label", "labels")
# Padding is done batch by batch, to the length of the batch's longest tweet
collate = DataCollatorWithPadding(tokenizer)
encoder_loaders = {
    split: DataLoader(encoded[split], batch_size=32, shuffle=split == "train",
                      collate_fn=collate)
    for split in ("train", "validation", "test")}


class TransformerClassifier(lightning.LightningModule):
  """Pretrained encoder topped with a new classification head."""

  def __init__(self, name: str, learning_rate: float = 2e-5,
               n_classes: int = 3) -> None:
    super().__init__()
    self.save_hyperparameters()
    # The classification head is created with random weights
    self.model = AutoModelForSequenceClassification.from_pretrained(
        name, num_labels=n_classes)
    self.recalls = nn.ModuleDict({
        f"{stage}_recall": MulticlassRecall(num_classes=n_classes, average="macro")
        for stage in ("val", "test")})

  def _step(self, batch: dict[str, torch.Tensor], stage: str) -> torch.Tensor:
    # Given batch["labels"], the model computes the cross-entropy itself
    outputs = self.model(**batch)
    batch_size = len(batch["labels"])
    self.log(f"{stage}_loss", outputs.loss, batch_size=batch_size, on_step=False,
             on_epoch=True, prog_bar=True)
    if stage != "train":
      recall = self.recalls[f"{stage}_recall"]
      recall(outputs.logits, batch["labels"])
      self.log(f"{stage}_recall", recall, batch_size=batch_size, on_step=False,
               on_epoch=True, prog_bar=True)
    return outputs.loss

  def training_step(self, batch: dict[str, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: dict[str, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: dict[str, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(self, batch: dict[str, torch.Tensor],
                   batch_index: int) -> torch.Tensor:
    return self.model(**batch).logits.argmax(dim=1)

  def configure_optimizers(self) -> torch.optim.Optimizer:
    # The whole model is fine-tuned: a small learning rate preserves what the
    # encoder learned during pre-training
    return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)

In [ ]:
# Your code here

### Solution

In [ ]:
encoder_model = TransformerClassifier(ENCODER)
encoder_checkpoint = ModelCheckpoint(monitor="val_recall", mode="max")
encoder_trainer = lightning.Trainer(
    max_epochs=2,
    accelerator="auto",
    devices=1,
    # Half-precision computations on GPU: twice as fast
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    logger=CSVLogger("logs", name="distilroberta"),
    callbacks=[encoder_checkpoint])
encoder_trainer.fit(encoder_model, encoder_loaders["train"],
                    encoder_loaders["validation"])
encoder_trainer.test(encoder_model, encoder_loaders["test"], ckpt_path="best")

encoder_predictions = torch.cat(encoder_trainer.predict(
    encoder_model, encoder_loaders["test"], ckpt_path="best")).cpu().numpy()
show_evaluation(truths, encoder_predictions, "Fine-tuned DistilRoBERTa")
print("Macro recall on the first 1,000 test tweets")
for name, model_predictions in [("LSTM", predictions),
                                ("Fine-tuned DistilRoBERTa", encoder_predictions),
                                ("cardiffnlp's RoBERTa", roberta_predictions)]:
  score = recall_score(truths[:1000], model_predictions[:1000], average="macro")
  print(f"{name}: {score:.3f}")

Usually, the encoder fine-tuned in a few minutes clearly beats the LSTM, and gets close to cardiffnlp's RoBERTa without matching it: the latter is bigger and was pretrained on millions of tweets, whose vocabulary and phrasing it knows. Fine-tuning a general-purpose pretrained encoder is the standard approach when no model exists yet for the task: its representations of words and sentences, learned from large amounts of text, make up for the small amount of labeled data.